# Lab 13: Read Text (OCR) and QR Codes

**Duration:** 3 hours. **Level:** beginner. **Work mode:** pairs or individual.

| Student name | Station no. | Date |
|---|---|---|
| | | |

## 1. Overview
Not everything a camera sees is an object. Labels, signs, parcels, tickets and name cards carry **information in writing**: printed words, numbers, QR codes and barcodes. In this lab the Jetson **reads** them. You build a **QR attendance checker** (a student shows a QR code and the program marks the name present) and an **OCR reader** that finds a date and numbers in a printed label. OCR means *optical character recognition*, reading letters from a picture.

**By the end of this lab you can:**
1. Create a QR code and explain what is stored in it.
2. Read QR codes (and barcodes) from the live camera and check them against a list.
3. Prepare a picture so text can be read better (grey, scale, threshold).
4. Read text with OCR and read the confidence of each word.
5. Pick useful data out of text with a pattern (regular expression) and save it in a log.

**How this notebook works.** Run a cell with **Shift + Enter**, from the top. Type your answers in the cells that say *Your answers*. The live video appears under the cell that starts it. Press **Ctrl + S** often.

> **Privacy.** A QR code or a label can hold a name or a number. Use only the codes you make in this lab and print or show them only to your classmates. Do not scan the cards of other people.

## 2. Background

**QR codes and barcodes.** A QR code is a square picture that stores up to a few hundred characters of text (a name, a web address, a number). The three big squares in the corners tell the reader where the code is and how it is turned, so a QR code can be read from any angle. A barcode on a product stores only a number, in black and white lines. Reading a code is **not** machine learning: it is a fixed method (OpenCV does it), so it is fast and exact, and it either works or it does not.

**OCR.** Reading printed letters is harder. The program must find the lines of text, cut out each letter and decide which character it is. This lab uses **Tesseract**, a free OCR program that runs on the CPU. It gives you every word with a **confidence** from 0 to 100. A low confidence means the program is not sure.

| | QR or barcode | OCR text |
|---|---|---|
| Stored data | Text chosen by you, with error correction | Whatever is written |
| Method | Fixed pattern finder | Recognises letters |
| Speed | Fast, every picture | Slow (about half a second or more), so you press a button |
| Fails when | The code is blurry, small, shiny or cut | The letters are small, blurry, curly or on a busy background |

**Getting better OCR.** Tesseract works best with dark letters on a plain light background, about 30 pixels tall. You prepare the picture with the tools of Lab 6:

| Step | Why |
|---|---|
| Grey (`cv2.cvtColor`) | Colour does not help to read letters |
| Scale up x2 (`cv2.resize`) | Small letters become big enough |
| Threshold (`cv2.adaptiveThreshold`) | Makes letters black and the page white, also under uneven light |

**Patterns in text.** Often you do not need all the text, only a date or a number. A **regular expression** (regex) describes a pattern, for example `\d{2}/\d{2}/\d{4}` means "two digits, a slash, two digits, a slash, four digits". Python's `re.findall` finds every match.

**Change one thing at a time.** If you break the notebook, close it and run `import labview; labview.reset(13)` in a cell of a new notebook (File, New, Notebook) for a fresh copy.

---
# Part A: Prepare (30 min)

### Step 1: Install the Reading Programs
OCR needs the program **Tesseract**, and barcodes need the library **zbar**. Setup of a new Jetson installs both. This cell checks them. If one is missing, it tells you what to do.

In [ ]:
import glob
import os
import shutil
import subprocess
import sys

print("This notebook is saved in:", os.getcwd())
print("tesseract:", shutil.which("tesseract") or "MISSING")
print("zbar library:", "found" if glob.glob("/usr/lib/*/libzbar.so*") else "MISSING")

If something says `MISSING`, install it now. The first cell asks for your password (it is not shown). The second cell installs the programs (about 1 minute, it needs the internet).

In [ ]:
import labview
labview.sudo_login()

In [ ]:
!sudo apt install -y tesseract-ocr libzbar0

### Step 2: Install the Python Parts
`pytesseract` lets Python talk to Tesseract. `pyzbar` reads barcodes with zbar. `qrcode` makes the QR pictures when the OpenCV of the Jetson cannot (the cell installs it by itself).

In [ ]:
for package, module in (("pytesseract", "pytesseract"), ("pyzbar", "pyzbar")):
    try:
        __import__(module)
        print(package, "is installed.")
    except ImportError:
        print("Installing", package, "...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)
        print(package, "is installed.")

### Step 3: Plan Your Reader
Think about where reading codes and text helps.

| Idea | Reads | Data in the code or text |
|---|---|---|
| Class attendance | QR code on a card | Student ID |
| Parcel sorting | QR code or barcode | Parcel number and place |
| Shop shelf | Barcode | Product number |
| Medicine or food check | Printed label (OCR) | Expiry date |
| Door sign | Printed sign (OCR) | Room name |

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Idea you chose:
- What is read (code or text) and where is the camera?:
- What should the program do after it reads the data?:


---
# Part B: QR Attendance (50 min)

### Step 1: The Settings
`ROSTER` is the list of people. The key is the text stored in the QR code, the value is the name that is shown. Add your own team members. `COOLDOWN` is the number of seconds before the same code is marked again (the log must not fill with the same person many times).

In [ ]:
# ---------------- Settings you can change ----------------
ROSTER = {"DKA4-AMINAH": "Aminah", "DKA4-AMIRUL": "Amirul", "DKA4-AMIRA": "Amira", "DKA4-AMEERA": "Ameera"}
COOLDOWN = 10            # seconds before the same code is marked again
# ----------------------------------------------------------
print("People in the list:", len(ROSTER))

### Step 2: Make the QR Codes
The cell makes one QR code picture for each person with `labview.qr_picture`. The pictures are saved as `qr_<name>.png`. Look at them in the cell output. To use them, open the file in the file browser on the left, right-click it and choose **Download**, then show it on a phone, or print it. Each QR code is only a picture of the text.

In [ ]:
import csv
import re
import time
from datetime import datetime

import labview
import cv2
import numpy as np
from IPython.display import Image, display


def make_qr(text, size=6):
    """A QR code picture (white border included) for this text."""
    return labview.qr_picture(text, size)       # OpenCV, or the qrcode package when OpenCV is old


for text, name in ROSTER.items():
    picture = make_qr(text)
    cv2.imwrite(f"qr_{name.lower()}.png", picture)
    print(f"{name}: stores the text {text}")
    display(Image(filename=f"qr_{name.lower()}.png", width=160))

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- How many characters are stored in the QR code of your name?:
- Does the picture contain your name, or only the text that you gave?:


### Step 3: Read a Code from the Picture
Before the live video, read one saved code. `labview.read_qr` gives a list with one `(text, corners)` for each code in the picture: the text and the four corner points. It uses `pyzbar`, because the OpenCV of this Jetson can find a code but cannot read it. If there is no code, the list is empty.

In [ ]:
first = sorted(glob.glob("qr_*.png"))[0]
picture = cv2.imread(first)
found = labview.read_qr(picture)                                   # a list: one (text, corners) for each code
text, corners = found[0] if found else ("", None)
print("File:", first)
print("Text read:", repr(text))
print("Corner points:", None if corners is None else corners.tolist())

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Text read from the picture:
- What are the corner points used for in the live program?:


### Step 4: The Attendance Program
The program reads every QR code in the picture and checks the text against `ROSTER`. A known code is marked **present** (green) and written to `attendance.csv` and `attendance.xlsx`. An unknown code is shown in red and not logged.

| Function | What it does |
|---|---|
| `read_codes(frame)` | Finds all QR codes in the picture. Returns a list of (text, corner points). |
| `mark(text)` | Marks a known person present. Returns `True` when a new line was logged, `False` when the person was marked less than `COOLDOWN` seconds ago. |
| `log_attendance(code, name)` | Adds a line to the log files |
| `process(frame, state)` | Reads, checks, draws. The live view calls it for every picture. |

In [ ]:
run = {"last": {}, "note": "", "read": False, "text": "", "warned": False}


def read_codes(frame):
    """All QR codes in the picture: a list of (text, corners). The text is empty if a code could not be read."""
    return labview.read_qr(frame)


def write_excel():
    """Rewrite attendance.xlsx from attendance.csv, so it opens in Excel with a header."""
    try:
        from openpyxl import Workbook
        from openpyxl.styles import Font
    except ImportError:
        if not run["warned"]:
            run["warned"] = True
            print("attendance.xlsx needs openpyxl. attendance.csv works in Excel too. Ask your instructor.")
        return
    book = Workbook()
    sheet = book.active
    sheet.title = "Attendance"
    with open("attendance.csv", newline="") as f:
        for row in csv.reader(f):
            sheet.append(row)
    for cell in sheet[1]:
        cell.font = Font(bold=True)
    for column, width in zip("ABC", (20, 18, 14)):
        sheet.column_dimensions[column].width = width
    book.save("attendance.xlsx")


def log_attendance(code, name):
    first = not os.path.exists("attendance.csv")
    with open("attendance.csv", "a", newline="") as f:
        writer = csv.writer(f)
        if first:
            writer.writerow(["time", "name", "code"])
        writer.writerow([datetime.now().strftime("%Y-%m-%d %H:%M:%S"), name, code])
    write_excel()


def mark(text, now=None):
    """Mark a known person present. True = a new line was logged."""
    now = time.time() if now is None else now
    if text not in ROSTER:
        return False
    if now - run["last"].get(text, -1e9) < COOLDOWN:
        return False
    run["last"][text] = now
    log_attendance(text, ROSTER[text])
    return True


def process(frame, state):
    shown = frame.copy()
    for text, corners in read_codes(frame):
        known = text in ROSTER
        colour = (0, 200, 0) if known else (0, 0, 255)
        cv2.polylines(shown, [corners.reshape(-1, 1, 2)], True, colour, 3)
        label = ROSTER[text] + " present" if known else ("unknown code" if text else "cannot read")
        x, y = corners[:, 0].min(), max(corners[:, 1].min() - 10, 20)
        cv2.putText(shown, label, (int(x), int(y)), cv2.FONT_HERSHEY_SIMPLEX, 0.8, colour, 2)
        if mark(text):
            run["note"] = f"{ROSTER[text]} marked at {datetime.now().strftime('%H:%M:%S')}"
    cv2.putText(shown, f"{state['fps']:.0f} FPS", (shown.shape[1] - 110, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
    if run["note"]:
        cv2.putText(shown, run["note"], (10, shown.shape[0] - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    return shown


print("The program is ready.")

### Step 5: Run the Attendance Checker
Run the cell. Show a QR code (on a phone screen or on paper) to the camera. Hold it still, flat and about 20 to 30 cm from the lens. Watch the coloured frame and the name. Try a code that is not in the list: make one with the cell below. Press **Stop** when you are done.

In [ ]:
run["last"].clear()
run["note"] = ""
live = labview.live(process)

To test an unknown code, run this cell. It makes `qr_stranger.png` with a text that is not in the `ROSTER`. Open it from the file browser and show it to the camera.

In [ ]:
cv2.imwrite("qr_stranger.png", make_qr("DKA4-STRANGER"))
display(Image(filename="qr_stranger.png", width=160))

### Step 6: See the Results
Run this cell after you pressed Stop. To open the Excel file, use the file browser on the left, right-click `attendance.xlsx`, choose **Download** and open it on your laptop.

In [ ]:
def show_results():
    for name in ("attendance.csv", "attendance.xlsx"):
        print(f"{name:<16}", "found" if os.path.exists(name) else "not created yet (it appears at the first marked person)")
    if os.path.exists("attendance.csv"):
        print()
        with open("attendance.csv") as f:
            print(f.read())


show_results()

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- FPS shown on the screen:
- From how far away could the camera read a QR code on your phone?:
- What happened when you tilted the code or covered a corner?:
- What did the program show for the stranger code?:
- First line of the log after the header:


---
# Part C: Read Text with OCR (50 min)

### Step 1: OCR on a Picture
First read a picture you control. The cell makes a picture of a label with `cv2.putText`, so you know the right answer, and reads it with Tesseract.

In [ ]:
import pytesseract

label = np.full((200, 520, 3), 255, np.uint8)
cv2.putText(label, "MILK 1L", (20, 70), cv2.FONT_HERSHEY_SIMPLEX, 1.6, (0, 0, 0), 4)
cv2.putText(label, "EXP 14/03/2027", (20, 140), cv2.FONT_HERSHEY_SIMPLEX, 1.3, (0, 0, 0), 3)
cv2.putText(label, "Lot 4821", (20, 185), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 0), 2)
cv2.imwrite("label.png", label)
display(Image(filename="label.png"))

text = pytesseract.image_to_string(label)
print("Text read:")
print(text)

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Did OCR read all three lines correctly?:
- Which characters were wrong (if any)?:


### Step 2: Confidence of Each Word
`image_to_data` gives each word with a **box** and a **confidence**. The cell prints the words and keeps only the words with a confidence of at least `MIN_CONF`.

In [ ]:
MIN_CONF = 50            # words below this confidence (0 to 100) are ignored


def read_words(img):
    """Words found by OCR: a list of (word, confidence, x, y, w, h). Words with a low confidence are dropped."""
    data = pytesseract.image_to_data(img, output_type=pytesseract.Output.DICT)
    words = []
    for i, word in enumerate(data["text"]):
        if word.strip() and float(data["conf"][i]) >= MIN_CONF:
            words.append((word.strip(), float(data["conf"][i]), data["left"][i], data["top"][i], data["width"][i], data["height"][i]))
    return words


for word, conf, x, y, w, h in read_words(label):
    print(f"{word:<12} confidence {conf:5.1f}   box ({x}, {y}, {w}, {h})")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Which word has the lowest confidence?:
- What happens to the list when you set MIN_CONF = 90?:


### Step 3: Prepare the Picture
Real labels are not perfect. The function `prepare` turns a picture into black letters on white, scaled up. The cell makes a hard picture (grey letters, uneven light and noise), reads it **before** and **after** the preparation, and shows the difference.

In [ ]:
def prepare(img, scale=2):
    """Grey, bigger, and black-on-white with a threshold that adapts to uneven light."""
    grey = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img
    grey = cv2.resize(grey, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
    return cv2.adaptiveThreshold(grey, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, 15)


rng = np.random.default_rng(1)
hard = (label * 0.45 + 90).astype(np.uint8)                      # grey letters on a grey page
gradient = np.tile(np.linspace(-50, 50, hard.shape[1]), (hard.shape[0], 1))[..., None]
hard = np.clip(hard + gradient + rng.normal(0, 12, hard.shape), 0, 255).astype(np.uint8)   # uneven light and noise
cv2.imwrite("label_hard.png", hard)
display(Image(filename="label_hard.png"))

print("Before preparing:", repr(pytesseract.image_to_string(hard).strip()))
print("After preparing: ", repr(pytesseract.image_to_string(prepare(hard)).strip()))

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Which text did OCR read before and after the preparation?:
- Which of the three steps (grey, scale, threshold) do you think helped the most? Why?:


### Step 4: Pick the Data out of the Text
The function `find_data` uses regular expressions to take out the **date** and the **numbers** and ignores the rest.

In [ ]:
def find_data(text):
    """The dates (dd/mm/yyyy) and the numbers of 3 or more digits found in a text."""
    dates = re.findall(r"\b\d{2}/\d{2}/\d{4}\b", text)
    numbers = [n for n in re.findall(r"\b\d{3,}\b", text) if n not in "".join(dates)]
    return dates, numbers


dates, numbers = find_data(pytesseract.image_to_string(prepare(label)))
print("Dates:  ", dates)
print("Numbers:", numbers)

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Dates and numbers found:
- What does `\\d{2}` mean in the pattern?:
- Why is the pattern better than reading every line yourself?:


### Step 5: Read a Real Label Live
The program reads text only when you press **Read text**, because OCR is slow. Hold a printed label, a sheet of text or a box with writing in front of the camera, keep it still and flat, and press the button. The words are drawn in boxes (green: sure, orange: less sure) and the dates and numbers are saved in `ocr_log.csv`.

In [ ]:
def want_read(state):
    run["read"] = True                       # the button only sets a flag; process does the slow work


def log_ocr(dates, numbers, text):
    first = not os.path.exists("ocr_log.csv")
    with open("ocr_log.csv", "a", newline="") as f:
        writer = csv.writer(f)
        if first:
            writer.writerow(["time", "dates", "numbers", "text"])
        writer.writerow([datetime.now().strftime("%Y-%m-%d %H:%M:%S"), " ".join(dates), " ".join(numbers), " ".join(text.split())[:200]])


def process_ocr(frame, state):
    shown = frame.copy()
    if run["read"]:
        run["read"] = False
        started = time.time()
        small = prepare(frame)
        words = read_words(small)
        run["words"] = [(w, c, x // 2, y // 2, ww // 2, hh // 2) for w, c, x, y, ww, hh in words]      # back to the size of the picture
        run["text"] = " ".join(w[0] for w in words)
        dates, numbers = find_data(run["text"])
        run["note"] = f"{len(words)} words in {time.time() - started:.1f} s | dates {dates} numbers {numbers}"
        log_ocr(dates, numbers, run["text"])
    for word, conf, x, y, w, h in run.get("words", []):
        colour = (0, 200, 0) if conf >= 75 else (0, 160, 255)
        cv2.rectangle(shown, (x, y), (x + w, y + h), colour, 2)
        cv2.putText(shown, word, (x, max(y - 4, 12)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, colour, 1)
    cv2.putText(shown, f"{state['fps']:.0f} FPS", (shown.shape[1] - 110, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
    if run["note"]:
        cv2.putText(shown, run["note"][:75], (10, shown.shape[0] - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
    return shown


print("The OCR program is ready.")

In [ ]:
run.update(read=False, words=[], text="", note="")
live = labview.live(process_ocr, actions={"Read text": want_read})

After you pressed Stop, see what was read:

In [ ]:
print("Last text read:")
print(run["text"])
print()
if os.path.exists("ocr_log.csv"):
    with open("ocr_log.csv") as f:
        print(f.read())

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Time that one reading took (shown at the bottom of the picture):
- Which text was read well and which badly?:
- What changed when you moved the label closer, tilted it, or changed the light?:
- Which dates or numbers did the program find?:


---
# Part D: Make It Your Own (25 min)
Add **at least one improvement** from the list, or one of your own. Change the code in the cells above, run that cell again, then run the live cell again.

- Make QR codes with your own texts (for example a room name or a parcel number) and a `ROSTER` for a parcel station.
- Read **barcodes**: `from pyzbar.pyzbar import decode`, then `decode(frame)` gives objects with `.type`, `.data` and `.rect`. Draw a box around each one.
- Count the people marked present and show `present: 3 of 4` on the screen.
- Show the names of the people who are **not** present yet.
- Read text on a second language by `pytesseract.image_to_string(img, lang="eng")` (other languages need their data installed).
- Make `find_data` find another pattern, for example a phone number or a price like `RM 12.50`.

> **Tip.** Change one thing at a time and run it after each change. To keep a copy of a cell before a big change, copy the cell (select it, press **C**, then **V**) and change the copy.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Improvement you chose:
- Which lines did you change?:
- Did it work the first time? What went wrong?:


---
# Part E: Test and Present (25 min)

### Step 1: Test
Delete the old logs, then test these situations and write what happened.

In [ ]:
for name in ("attendance.csv", "attendance.xlsx", "ocr_log.csv"):
    if os.path.exists(name):
        os.remove(name)
print("Clean. Files left:", sorted(glob.glob("attendance*") + glob.glob("ocr_log*")))

| Test | Situation | Expected | What happened |
|---|---|---|---|
| 1 | A known QR code on a phone, 25 cm away | Name shown, one log line | |
| 2 | The same code again after 3 seconds | No new log line (cooldown) | |
| 3 | The same code 15 seconds later | A new log line | |
| 4 | A QR code that is not in the list | Red frame, no log line | |
| 5 | The QR code far away (60 cm) | Read or not? Write the distance at which it fails | |
| 6 | A printed label, flat, good light | Words in green boxes | |
| 7 | The same label, tilted or in dim light | Lower confidence or wrong words | |

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Test 1 to 4: were all four as expected?:
- Test 5: the distance where the code stopped to be read:
- Test 6 and 7: how did the confidence change?:
- MIN_CONF and COOLDOWN values that worked best:


### Step 2: Report and Present
Fill in the report. Then give a **3-minute demonstration**: make a QR code, mark a person present, show a rejected code, read a label with OCR and show the logs.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Idea and what the program reads:
- FPS shown and the time of one OCR reading:
- Power mode used (run  !sudo nvpmodel -q  in a cell, after  labview.sudo_login()  ):
- Your improvement:
- One problem you found and how you fixed it:
- One idea to make it better in future:


> **Where next.** So far the models know the 80 classes of the course. In Lab 14 you teach a model **new objects that you choose yourself**.

## 4. Check Task
Show your instructor. The instructor initials each line when it is done correctly.

| No. | Task to demonstrate | Initials |
|---|---|---|
| 1 | QR codes are made and the text stored in each one is explained | |
| 2 | The live checker marks a known person present and logs one line | |
| 3 | An unknown code is shown in red and not logged | |
| 4 | The cooldown stops the same person from being logged again at once | |
| 5 | OCR reads the label picture and the confidence of each word is shown | |
| 6 | The prepared picture reads better than the original (Part C Step 3) | |
| 7 | The live OCR finds a date or a number from a real label | |
| 8 | At least one improvement works and you explain the code you changed | |
| 9 | The test table is complete and the 3-minute presentation is given | |

## 5. Assessment Questions
Answer in one or two sentences.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- 1. Is reading a QR code machine learning? Explain why it is fast and exact.:
- 2. What is stored in a QR code? Does the picture contain a name by itself?:
- 3. Why does the attendance program use a cooldown?:
- 4. Why is OCR run only when you press a button, and QR reading on every picture?:
- 5. Name the three steps that prepare a picture for OCR, and say what each one does.:
- 6. What does the confidence of a word tell you? What is the cost of a high `MIN_CONF`?:
- 7. Write a regular expression for a time such as 09:45 (two digits, a colon, two digits).:
- 8. A QR code on a shiny sticker is not read. Name two things that could help.:
- 9. Which is a good use of OCR? (a) Finding a face (b) Reading the expiry date on a package (c) Counting cars (d) Measuring a temperature:


## 6. Troubleshooting

| Symptom | Likely cause | What to try |
|---|---|---|
| `tesseract: MISSING` or `TesseractNotFoundError` | Tesseract is not installed | Run the three install cells of Part A (password, then `apt install`) |
| `No module named 'pytesseract'` | Step 2 of Part A was not run | Run Step 2 of Part A |
| `Unable to find zbar` (barcodes) | `libzbar0` is not installed | Run the `apt install` cell of Part A |
| `AttributeError ... QRCodeEncoder` | The OpenCV on this Jetson is old | Ask your teacher. As a workaround run `!pip install qrcode` and use `qrcode.make(text)` to make the picture. |
| The QR code is not read | Too small, blurry, tilted a lot, shiny, or cut at an edge | Move to 20 to 30 cm, hold still, turn off the shine of the phone, show the whole code with its white border |
| The name is wrong or `unknown code` | The text in the code is not in `ROSTER` | Check the spelling and capitals of the keys of `ROSTER` and make the codes again |
| OCR gives nonsense | Small letters, busy background, bad light | Use a plain label, bigger letters, good light. Check `prepare` with different `scale` values. |
| `Cannot open the camera` | Another program uses the camera, or the cable is loose | Press Stop in other live cells, shut down other kernels, ask your teacher to check your camera on the teacher page |
| The Read text button does nothing | The live cell was not run, or the picture is not updating | Run the cell that starts the live view again, then press the button |
| The notebook looks broken | You changed or deleted too much | Close it and run `import labview; labview.reset(13)` in a cell of a new notebook (File, New, Notebook). You get a fresh copy. |

## 7. Further Reading
- OpenCV, QR code detection: https://docs.opencv.org/4.x/de/dc3/classcv_1_1QRCodeDetector.html
- Tesseract OCR: https://github.com/tesseract-ocr/tesseract
- pytesseract: https://github.com/madmaze/pytesseract
- pyzbar: https://github.com/NaturalHistoryMuseum/pyzbar
- Python documentation, re (regular expressions): https://docs.python.org/3/library/re.html